Starter

In [2]:
!pip install mysql-connector-python

   ---------------------------------------- 0.0/16.1 MB ? eta -:--:--
   - -------------------------------------- 0.5/16.1 MB 2.8 MB/s eta 0:00:06
   -- ------------------------------------- 1.0/16.1 MB 3.4 MB/s eta 0:00:05
   -- ------------------------------------- 1.0/16.1 MB 3.4 MB/s eta 0:00:05
   --- ------------------------------------ 1.6/16.1 MB 1.8 MB/s eta 0:00:09
   ------ --------------------------------- 2.6/16.1 MB 2.5 MB/s eta 0:00:06
   ------- -------------------------------- 3.1/16.1 MB 2.6 MB/s eta 0:00:06
   --------- ------------------------------ 3.9/16.1 MB 2.8 MB/s eta 0:00:05
   ----------- ---------------------------- 4.7/16.1 MB 2.9 MB/s eta 0:00:04
   ------------- -------------------------- 5.2/16.1 MB 2.9 MB/s eta 0:00:04
   -------------- ------------------------- 5.8/16.1 MB 2.7 MB/s eta 0:00:04
   -------------- ------------------------- 6.0/16.1 MB 2.7 MB/s eta 0:00:04
   ---------------- ----------------------- 6.8/16.1 MB 2.7 MB/s eta 0:00:04
   ---

In [7]:
import random
import copy
import mysql.connector

# =========================================
# 0. เชื่อมต่อฐานข้อมูล
# =========================================
def connect_db():
    """
    ปรับแก้ host, user, password, database ให้ตรงกับของคุณ
    """
    conn = mysql.connector.connect(
        host="localhost",
        user="root",
        password="",
        database="rov"
    )
    return conn

# =========================================
# 1. Parameter Setting
# =========================================
POPULATION_SIZE = 50
MAX_GENERATIONS = 100
CROSSOVER_RATE = 0.8
MUTATION_RATE = 0.05
ELITISM = True

# Budget ตาม Phase
EARLY_BUDGET = 2700
MID_BUDGET = 7500
LATE_BUDGET = 14000

CHROMOSOME_LENGTH = 6  # 6 ชิ้น

# =========================================
# 2. Load ข้อมูลพื้นฐานจาก DB
# =========================================

def load_item_data():
    """
    ดึงข้อมูลไอเทมทั้งหมดจากตาราง items
    ควรมีคอลัมน์ ItemID, ItemType, Cost, PhysicalAttack, MagicPower, Defense, HP, CDR, ฯลฯ
    (แล้วแต่โครงสร้างจริงของคุณ)
    """
    conn = connect_db()
    cursor = conn.cursor(dictionary=True)

    query = "SELECT * FROM items"
    cursor.execute(query)
    items = cursor.fetchall()

    cursor.close()
    conn.close()

    # แปลงเป็น dict โดยมี key = ItemID
    item_dict = {}
    for it in items:
        item_id = it["ItemID"]
        item_dict[item_id] = it  # เก็บทั้ง row ไว้เลย
    return item_dict

def load_hero_data(hero_id):
    """
    ดึงข้อมูลฮีโร่ (เช่น Class, Lane ได้จากตาราง heroes)
    ดึงข้อมูลสเตตัสพื้นฐาน (herostats) ถ้าจำเป็น
    หรือต้องการข้อมูลเฉพาะ hero_id
    """
    conn = connect_db()
    cursor = conn.cursor(dictionary=True)

    # ตัวอย่าง: ดึงข้อมูลจาก heroes
    query_hero = "SELECT * FROM heroes WHERE HeroID = %s"
    cursor.execute(query_hero, (hero_id,))
    hero_info = cursor.fetchone()

    cursor.close()
    conn.close()

    return hero_info  # เช่น {HeroID:'H001', Hero_Name:'Airi', First_Class:'Assassin', ...}

def load_hero_skills(hero_id):
    """
    ดึงข้อมูล heroskills เพื่อดู RecommendItemType
    เช่น SkillID, HeroID, SkillName, RecommendItemType เป็นต้น
    """
    conn = connect_db()
    cursor = conn.cursor(dictionary=True)

    query_skill = "SELECT * FROM heroskills WHERE HeroID = %s"
    cursor.execute(query_skill, (hero_id,))
    skills = cursor.fetchall()

    cursor.close()
    conn.close()

    return skills  # list ของ dict

# =========================================
# 3. Representation / Population Initialization
# =========================================

def create_random_chromosome(item_pool, force_items, ban_items):
    """
    สร้างโครโมโซมแบบสุ่ม (6 ไอเทม) โดย
    1) ใส่ force_items ก่อน
    2) หลีกเลี่ยง ban_items
    3) ไม่ซ้ำกัน (ถ้า RoV ไม่อนุญาตออกไอเทมซ้ำ)
    """
    chromosome = []

    # 1) ใส่ force items
    forced_list = list(force_items)
    for fi in forced_list:
        if fi not in chromosome and fi not in ban_items:
            chromosome.append(fi)

    # 2) สุ่มไอเทมที่เหลือจนกว่าจะครบ 6 ชิ้น
    while len(chromosome) < CHROMOSOME_LENGTH:
        candidate = random.choice(item_pool)
        if (candidate not in ban_items) and (candidate not in chromosome):
            chromosome.append(candidate)

    # กรณีสุ่มได้ซ้ำจนคับไม่ถึง 6 ชิ้น อาจต้องวนซ้ำเพิ่ม
    if len(chromosome) < CHROMOSOME_LENGTH:
        # ลองวนอีกสักรอบ
        for _ in range(1000):  # ป้องกัน infinite loop
            if len(chromosome) >= CHROMOSOME_LENGTH:
                break
            candidate = random.choice(item_pool)
            if (candidate not in ban_items) and (candidate not in chromosome):
                chromosome.append(candidate)

    # ถ้ายังไม่ได้ 6 ชิ้นจริง ๆ (กรณี ban เยอะมาก) อาจปล่อยผ่าน
    return chromosome[:CHROMOSOME_LENGTH]

def initialize_population(size, item_pool, force_items, ban_items):
    population = []
    for _ in range(size):
        chromo = create_random_chromosome(item_pool, force_items, ban_items)
        population.append(chromo)
    return population

# =========================================
# 4. Fitness Function (หลัก) 
# =========================================

def calculate_fitness(chromosome, hero_id, lane, item_data, hero_info, hero_skills):
    """
    ประเมินคะแนน Fitness ของชุดไอเทม 6 ชิ้น
    1) รวมสเตตัสไอเทม -> ถ่วงน้ำหนักตาม Class
    2) ตรวจสอบ Lane (เช่น Jungle -> ต้องมีไอเทมป่า, Support -> ต้องมีไอเทมซัพพอร์ต ฯลฯ)
    3) RecommendedItemType จากสกิล -> ถ้ามีไอเทมตรงประเภท -> +คะแนน
    4) Early/Mid/Late Budget
    5) ฯลฯ
    """

    # ----------------------------------
    # 4.1 ข้อมูลฮีโร่ เช่น Class
    # ----------------------------------
    # สมมติ heroes มี First_Class, Second_Class
    # เลือกใช้ First_Class เป็นหลัก (หรือตรวจสอบว่าเป็นสายไหน)
    hero_class = hero_info["First_Class"]  # ex: 'Assassin'
    # ตัวอย่างการกำหนดน้ำหนัก
    # คุณควรปรับค่าจริงตามงานวิจัย/การออกแบบ
    weight_table = {
        "Fighter":  {"PATK":0.3,"AP":0.0,"DEF":0.2,"HP":0.2,"CDR":0.2,"CRIT":0.1,"MS":0.0},
        "Tank":     {"PATK":0.1,"AP":0.0,"DEF":0.4,"HP":0.4,"CDR":0.0,"CRIT":0.0,"MS":0.1},
        "Assassin": {"PATK":0.4,"AP":0.1,"DEF":0.1,"HP":0.1,"CDR":0.3,"CRIT":0.1,"MS":0.0},
        "Mage":     {"PATK":0.0,"AP":0.5,"DEF":0.1,"HP":0.1,"CDR":0.2,"CRIT":0.0,"MS":0.1},
        "Carry":    {"PATK":0.4,"AP":0.0,"DEF":0.1,"HP":0.1,"CDR":0.1,"CRIT":0.3,"MS":0.0},
        "Support":  {"PATK":0.1,"AP":0.3,"DEF":0.2,"HP":0.3,"CDR":0.3,"CRIT":0.0,"MS":0.1}
    }
    if hero_class not in weight_table:
        hero_class = "Fighter"  # fallback

    # ----------------------------------
    # 4.2 รวมค่าสเตตัสจากไอเทม 
    # (PATK, AP, DEF, HP, CDR, CRIT, MS etc.)
    # ----------------------------------
    total_patk = 0
    total_ap = 0
    total_def = 0
    total_hp = 0
    total_cdr = 0
    total_crit = 0
    total_ms = 0
    total_cost = 0

    for item_id in chromosome:
        if item_id not in item_data:
            continue  # กรณีข้อมูลไม่ครบ
        it = item_data[item_id]
        # ปรับชื่อคอลัมน์ตามฐานข้อมูลจริง
        total_cost += it.get("Cost", 0)
        total_patk += it.get("PhysicalAttack", 0)
        total_ap   += it.get("MagicPower", 0)
        total_def  += it.get("Defense", 0)
        total_hp   += it.get("HP", 0)
        total_cdr  += it.get("CDR", 0)
        total_crit += it.get("CritChance", 0)
        total_ms   += it.get("MoveSpeed", 0)

    # ----------------------------------
    # 4.3 การถ่วงน้ำหนักด้วย Hero Class
    # ----------------------------------
    # (อาจ normalize ก่อน หรือคูณตรง ๆ เลยก็ได้)
    wtable = weight_table[hero_class]
    score_stats = (
        wtable["PATK"] * total_patk +
        wtable["AP"]   * total_ap +
        wtable["DEF"]  * total_def +
        wtable["HP"]   * total_hp +
        wtable["CDR"]  * total_cdr +
        wtable["CRIT"] * total_crit +
        wtable["MS"]   * total_ms
    )

    # ----------------------------------
    # 4.4 ตรวจสอบ Lane
    # ----------------------------------
    # สมมติ:
    # - Jungle ต้องมีไอเทมที่ ItemType = 'Jungle' อย่างน้อย 1 ชิ้น
    # - Support ต้องมี ItemType = 'Support' อย่างน้อย 1 ชิ้น
    # - อื่น ๆ คุณอาจปรับเงื่อนไขได้
    lane_score = 0
    if lane == "Jungle":
        has_jungle_item = False
        for item_id in chromosome:
            if item_id in item_data:
                it_type = item_data[item_id].get("ItemType","")
                if it_type == "Jungle":
                    has_jungle_item = True
                    break
        if has_jungle_item:
            lane_score += 10
        else:
            lane_score -= 20  # ลงโทษหนักถ้าไม่มี
    elif lane == "Support":
        has_support_item = False
        for item_id in chromosome:
            if item_id in item_data:
                it_type = item_data[item_id].get("ItemType","")
                if it_type == "Support":
                    has_support_item = True
                    break
        if has_support_item:
            lane_score += 10
        else:
            lane_score -= 20

    # สำหรับ Off Lane / Mid / Dragon Lane คุณอาจกำหนดเงื่อนไขเสริมเอง

    # ----------------------------------
    # 4.5 RecommendedItemType จากสกิล
    # ----------------------------------
    skill_score = 0
    # ตัวอย่าง: ถ้าสกิลมี RecommendItemType = "MagicPen" แล้วใน chromosome มีไอเทม ItemType = "MagicPen" -> +คะแนน
    for sk in hero_skills:
        rec_type = sk.get("RecommendItemType", "")
        if not rec_type:
            continue
        # ตรวจสอบในโครโมโซมว่ามีไอเทมใด ItemType == rec_type หรือไม่
        for item_id in chromosome:
            if item_id in item_data:
                it_type = item_data[item_id].get("ItemType","")
                # เช็คแบบง่าย ๆ: ถ้า rec_type ตรงกับ it_type
                if it_type == rec_type:
                    skill_score += 5  # +5 ต่อ 1 ไอเทมที่แมตช์
                    # ไม่ต้อง break; ถ้าชุดนึงมีหลายชิ้นก็ + หลายรอบ

    # ----------------------------------
    # 4.6 คิดงบในช่วง Early/Mid/Late
    # ----------------------------------
    # แนวทาง: 
    #   - Early: พิจารณา 2 ชิ้นแรก => cost_early <= 2700
    #   - Mid:   4 ชิ้นแรก => cost_mid <= 7500
    #   - Late:  6 ชิ้น => cost_late <= 14000
    # แล้วเอาไปคิดคะแนนรวมกัน
    # (ตัวอย่างนี้จะคำนวณ Fitness แบบ multi-phase)
    cost_early = 0
    cost_mid   = 0
    cost_late  = total_cost

    # คำนวณ 2 ชิ้นแรก
    for i, item_id in enumerate(chromosome):
        if i < 2:  # early
            cost_early += item_data[item_id].get("Cost",0)
        if i < 4:  # mid
            cost_mid += item_data[item_id].get("Cost",0)

    # สมมติเราจัดคะแนนตาม Phase
    def phase_score(cost_val, budget):
        if cost_val <= budget:
            return 1.0  # ผ่าน
        else:
            # ถ้าเกิน อาจคิดเป็นสัดส่วน
            over = cost_val - budget
            return max(0, 1 - (over / budget))  # ถ้ายิ่งเกินเยอะ ยิ่งได้ 0

    early_factor = phase_score(cost_early, EARLY_BUDGET)
    mid_factor   = phase_score(cost_mid, MID_BUDGET)
    late_factor  = phase_score(cost_late, LATE_BUDGET)

    # ถ่วงน้ำหนัก
    w_e, w_m, w_l = 0.2, 0.3, 0.5  # ตัวอย่าง
    budget_score = (w_e*early_factor + w_m*mid_factor + w_l*late_factor) * 20
    # เอา *20 เป็น scaling factor เพื่อให้ได้ตัวเลขคะแนนรวม

    # ----------------------------------
    # 4.7 รวมคะแนนสุดท้าย
    # ----------------------------------
    fitness = 0.0
    fitness += score_stats   # คะแนนสเตตัส
    fitness += lane_score    # คะแนนเงื่อนไขเลน
    fitness += skill_score   # คะแนนจาก RecommendItemType
    fitness += budget_score  # คะแนนจากการไม่เกินงบ

    return fitness

# =========================================
# 5. Evaluate Population
# =========================================
def evaluate_population(population, hero_id, lane, item_data):
    """
    ประเมิน Fitness ให้โครโมโซมทุกตัว
    โดยจะโหลดข้อมูล hero, skills มาใช้
    (ป้องกันไม่ให้โหลดซ้ำ ๆ บ่อย ก็ได้)
    """
    hero_info = load_hero_data(hero_id)
    hero_skills = load_hero_skills(hero_id)

    fitness_values = []
    for chromo in population:
        fit = calculate_fitness(
            chromosome=chromo,
            hero_id=hero_id,
            lane=lane,
            item_data=item_data,
            hero_info=hero_info,
            hero_skills=hero_skills
        )
        fitness_values.append(fit)
    return fitness_values

# =========================================
# 6. Selection
# =========================================
def selection(population, fitness_values):
    """
    Tournament Selection (size=2)
    """
    new_population = []
    pop_size = len(population)
    for _ in range(pop_size):
        i1, i2 = random.sample(range(pop_size), 2)
        if fitness_values[i1] > fitness_values[i2]:
            winner = copy.deepcopy(population[i1])
        else:
            winner = copy.deepcopy(population[i2])
        new_population.append(winner)
    return new_population

# =========================================
# 7. Crossover
# =========================================
def crossover(parent1, parent2):
    """ One-point crossover """
    if random.random() < CROSSOVER_RATE:
        point = random.randint(1, CHROMOSOME_LENGTH-1)
        child1 = parent1[:point] + parent2[point:]
        child2 = parent2[:point] + parent1[point:]
        # ต้องตรวจสอบการซ้ำหรือ Ban ไหม
        # ตัวอย่างนี้ปล่อยให้ mutation แก้ไขต่อ
        return child1, child2
    else:
        return copy.deepcopy(parent1), copy.deepcopy(parent2)

# =========================================
# 8. Mutation
# =========================================
def mutation(chromosome, item_pool, ban_items):
    for i in range(len(chromosome)):
        if random.random() < MUTATION_RATE:
            candidate = random.choice(item_pool)
            # ตรวจสอบซ้ำ / ban
            if (candidate not in ban_items) and (candidate not in chromosome):
                chromosome[i] = candidate
    return chromosome

# =========================================
# 9. Replacement
# =========================================
def replacement(old_population, old_fitness, new_population, new_fitness):
    if not ELITISM:
        return new_population, new_fitness
    else:
        # หา best ตัวเก่า
        best_index = max(range(len(old_population)), key=lambda i: old_fitness[i])
        best_chromo = old_population[best_index]
        best_fit = old_fitness[best_index]

        # หา worst ตัวใน new_population
        worst_index = min(range(len(new_population)), key=lambda i: new_fitness[i])

        # แทนที่
        new_population[worst_index] = copy.deepcopy(best_chromo)
        new_fitness[worst_index] = best_fit
        return new_population, new_fitness

# =========================================
# 10. Main GA Loop
# =========================================
def run_genetic_algorithm(hero_id, lane, force_items, ban_items):
    """
    hero_id: ฮีโร่ที่เลือก
    lane: เลนของฮีโร่ เช่น "Jungle", "Support", ...
    force_items: set() หรือ list ของไอเทมบังคับ
    ban_items: set() หรือ list ของไอเทมที่แบน
    """

    # โหลดข้อมูลไอเทมทั้งหมด
    item_data = load_item_data()
    # สร้าง item_pool = รายการไอเทมทั้งหมดที่ไม่ถูก ban ล่วงหน้า
    # (ยกเว้น force_items ยังใช้ได้อยู่)
    item_pool = [it_id for it_id in item_data.keys() if it_id not in ban_items]

    # 1) Init Population
    population = initialize_population(POPULATION_SIZE, item_pool, force_items, ban_items)
    # 2) Evaluate
    fitness_values = evaluate_population(population, hero_id, lane, item_data)

    best_solution = None
    best_fitness = float("-inf")

    generation = 0
    while generation < MAX_GENERATIONS:
        # 3) Selection
        mating_pool = selection(population, fitness_values)
        # 4) Crossover
        new_population = []
        for i in range(0, len(mating_pool), 2):
            parent1 = mating_pool[i]
            if i+1 < len(mating_pool):
                parent2 = mating_pool[i+1]
            else:
                parent2 = mating_pool[0]  # ถ้าจำนวนเป็นคี่
            child1, child2 = crossover(parent1, parent2)
            new_population.append(child1)
            new_population.append(child2)

        # 5) Mutation
        for i in range(len(new_population)):
            new_population[i] = mutation(new_population[i], item_pool, ban_items)

        # Evaluate new population
        new_fitness = evaluate_population(new_population, hero_id, lane, item_data)

        # 6) Replacement
        population, fitness_values = replacement(population, fitness_values,
                                                 new_population, new_fitness)

        # เช็ค best
        curr_best_index = max(range(len(population)), key=lambda i: fitness_values[i])
        curr_best_fit = fitness_values[curr_best_index]
        if curr_best_fit > best_fitness:
            best_fitness = curr_best_fit
            best_solution = copy.deepcopy(population[curr_best_index])

        generation += 1

    return best_solution, best_fitness


# =========================================
# ตัวอย่างการทดสอบเรียกใช้งาน (main)
# =========================================
if __name__ == "__main__":
    # ตัวอย่างสมมติผู้ใช้เลือกฮีโร่ = "H001" Lane = "Jungle"
    hero_id_example = "H001"
    lane_example = "Jungle"

    # Force Item / Ban Item (สมมติ)
    force_list = {"I010", "I022"}  # บังคับ 2 ชิ้น
    ban_list   = {"I999"}         # แบน 1 ชิ้น

    best_build, best_fit = run_genetic_algorithm(
        hero_id=hero_id_example,
        lane=lane_example,
        force_items=force_list,
        ban_items=ban_list
    )

    print("Best Build:", best_build)
    print("Best Fitness:", best_fit)


Best Build: ['I044', 'I051', 'I042', 'I040', 'I048', 'I040']
Best Fitness: 800.0
